# MoCo Colab Run

This notebook clones the latest `method/moco-v2` branch, copies `dataset.zip` from Google Drive to Colab local SSD, writes all `./output` artifacts into a Drive-backed run folder, and runs MoCo pretraining, fine-tuning, and evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from datetime import datetime

REPO_URL = 'https://github.com/satvikkaul/SSL_Prostate_Cancer_Grading.git'
BRANCH = 'method/moco-v2'
PROJECT_DIR = '/content/SSL_Prostate_Cancer_Grading'

# ── Google Drive path ──────────────────────────────────────────────────────────
# The dataset was shared with you by a teammate. Use ONE of the options below
# depending on how the folder appears in your Drive after mounting:
#
#   Option A – Shared folder added to your "My Drive" (most common):
#     DRIVE_ROOT = '/content/drive/MyDrive/Prostate_SSL'
#
#   Option B – Google "Shared Drive" (Team Drive):
#     DRIVE_ROOT = '/content/drive/Shareddrives/Prostate_SSL'
#
# After running Cell 1 (drive.mount), open the file browser on the left in Colab
# (folder icon) and navigate to /content/drive to confirm which path is correct.
# ──────────────────────────────────────────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/Prostate_SSL'

DATASET_ZIP = f'{DRIVE_ROOT}/dataset.zip'
RUN_NAME = f'moco_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
DRIVE_RUN_DIR = f'{DRIVE_ROOT}/runs/{RUN_NAME}'
os.environ['MPLCONFIGDIR'] = '/tmp/mplconfig'

print('PROJECT_DIR  =', PROJECT_DIR)
print('DATASET_ZIP  =', DATASET_ZIP)
print('DRIVE_RUN_DIR=', DRIVE_RUN_DIR)


In [ ]:
%cd /content
!rm -rf "$PROJECT_DIR"
!git clone -b "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"
!git branch --show-current
!git log -1 --oneline

In [ ]:

# ── Fine-tune hyperparameter overrides (A100 GPU) ─────────────────────────────
# The repo's finetune_moco.py has conservative defaults (batch=8, 30+10 epochs).
# Since this notebook clones directly from GitHub, we patch the values here
# after cloning so you don't need push access to the repo.
# ─────────────────────────────────────────────────────────────────────────────
import re, pathlib

ft_path = pathlib.Path(PROJECT_DIR) / 'training/moco/finetune_moco.py'
src = ft_path.read_text()

replacements = {
    r'^BATCH_SIZE\s*=\s*\d+':     'BATCH_SIZE = 64        # A100: increased from 8',
    r'^EPOCHS_STAGE_1\s*=\s*\d+': 'EPOCHS_STAGE_1 = 50    # A100: increased from 30',
    r'^EPOCHS_STAGE_2\s*=\s*\d+': 'EPOCHS_STAGE_2 = 20    # A100: increased from 10',
}

for pattern, replacement in replacements.items():
    src = re.sub(pattern, replacement, src, flags=re.MULTILINE)

ft_path.write_text(src)
print('finetune_moco.py patched:')
for line in src.splitlines():
    if any(k in line for k in ['BATCH_SIZE', 'EPOCHS_STAGE_1', 'EPOCHS_STAGE_2']):
        if not line.strip().startswith('#'):
            print(' ', line)


In [ ]:
%cd "$PROJECT_DIR"
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!python --version
!nvidia-smi

In [ ]:
%cd "$PROJECT_DIR"
!test -f "$DATASET_ZIP" || (echo "Missing dataset zip at $DATASET_ZIP" && exit 1)
!rm -rf ./dataset
!cp "$DATASET_ZIP" ./dataset.zip
!unzip -q ./dataset.zip
!rm -f ./dataset.zip
!ls ./dataset | head

In [ ]:
%cd "$PROJECT_DIR"
!mkdir -p "$DRIVE_RUN_DIR/output"
!mkdir -p "$DRIVE_RUN_DIR/notebook_logs"
!rm -rf ./output
!ln -s "$DRIVE_RUN_DIR/output" ./output
!echo "$RUN_NAME" > "$DRIVE_RUN_DIR/notebook_logs/run_name.txt"
!pwd
!ls -ld ./output
!ls -ld "$DRIVE_RUN_DIR/output"

In [ ]:
%cd "$PROJECT_DIR"
import os

required_files = [
    './dataset/Train.csv',
    './dataset/Test.csv',
    './dataset/TrainSplit.csv',
    './dataset/Val.csv',
    './dataset/Pretrain_Manifest.csv',
]

missing = [path for path in required_files if not os.path.exists(path)]
if missing:
    print('Missing generated dataset files:', missing)
    !python data/setup.py
else:
    print('Dataset CSVs and manifest already present.')

## Pretrain

Edit `EPOCHS`, `BATCH_SIZE`, and `SAVE_FREQ` before running. This cell writes checkpoints directly under the Drive-backed `./output` symlink, so no extra copy step is required.

In [ ]:
%cd "$PROJECT_DIR"

# ── Pretrain hyperparameters (A100 GPU) ───────────────────────────────────────
EPOCHS = 100       # Full pretrain run; increase to 200 for a longer experiment
BATCH_SIZE = 256   # A100 (40 GB) comfortably handles 256; drop to 128 if OOM
SAVE_FREQ = 10     # Save a checkpoint to Drive every 10 epochs
RESUME = ''        # Leave empty for a fresh run.
                   # To resume a crashed run set to the last state checkpoint,
                   # e.g. './output/models/moco/state/moco_state-90'
# ─────────────────────────────────────────────────────────────────────────────

cmd = f'python training/moco/pretrain_moco.py --epochs {EPOCHS} --batch_size {BATCH_SIZE} --save_freq {SAVE_FREQ}'
if RESUME.strip():
    cmd += f' --resume {RESUME}'
print(cmd)
!$cmd


## Fine-Tune

This cell picks the latest saved `encoder_q_epoch*.weights.h5` checkpoint unless you override it.

In [ ]:
%cd "$PROJECT_DIR"
import glob

checkpoint_candidates = sorted(glob.glob('./output/models/moco/encoder_q_epoch*.weights.h5'))
if not checkpoint_candidates:
    raise FileNotFoundError('No MoCo encoder checkpoints found under ./output/models/moco/')

CHECKPOINT = checkpoint_candidates[-1]
print('Using checkpoint:', CHECKPOINT)
!python training/moco/finetune_moco.py --checkpoint "$CHECKPOINT"

## Evaluate

This uses `best_moco_overall.keras` by default. All evaluation artifacts are written into the Drive-backed `./output/moco/` folder.

In [ ]:
%cd "$PROJECT_DIR"
!python evaluation/moco/eval_moco.py

## Sync Runtime State to Drive

Use this after pretraining, fine-tuning, or evaluation if you want an explicit sync of generated runtime artifacts beyond the Drive-backed `./output` folder. This copies generated dataset CSVs/manifests, the notebook itself, and a small run summary into the current Drive run directory.

In [ ]:
%cd "$PROJECT_DIR"
!mkdir -p "$DRIVE_RUN_DIR/runtime_sync/dataset"
!mkdir -p "$DRIVE_RUN_DIR/runtime_sync/notebook"
!cp -f ./run_colab.ipynb "$DRIVE_RUN_DIR/runtime_sync/notebook/"
!for f in Train.csv Test.csv TrainSplit.csv Val.csv Pretrain_Manifest.csv; do if [ -f "./dataset/$f" ]; then cp -f "./dataset/$f" "$DRIVE_RUN_DIR/runtime_sync/dataset/"; fi; done
!git rev-parse HEAD > "$DRIVE_RUN_DIR/runtime_sync/commit.txt"
!find ./output -maxdepth 4 -type f | sort > "$DRIVE_RUN_DIR/runtime_sync/output_manifest.txt"
!echo "Synced runtime artifacts to: $DRIVE_RUN_DIR/runtime_sync"

In [ ]:
%cd "$PROJECT_DIR"
!echo "Run directory: $DRIVE_RUN_DIR"
!find ./output -maxdepth 3 -type f | sort | tail -n 40